In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader

# Same transforms as Day 4
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder("data/train", transform=train_transforms)
val_dataset   = datasets.ImageFolder("data/val",   transform=val_transforms)
train_loader  = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print("Data loaded!")
print("Classes:", train_dataset.classes)


Data loaded!
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [2]:
# Reload model architecture
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(1280, 6)
)

# Load the weights we trained on Day 4
model.load_state_dict(torch.load("clearbin_model.pth"))
print("Day 4 weights loaded!")

# Now UNFREEZE all layers for fine tuning
for param in model.parameters():
    param.requires_grad = True

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")
print("All layers unfrozen - fine tuning mode!")

Day 4 weights loaded!
Trainable parameters: 4,015,234
All layers unfrozen - fine tuning mode!


In [3]:
# Much smaller learning rate for fine tuning
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# Learning rate scheduler - reduces LR when progress stalls
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=2, factor=0.5
)

criterion = nn.CrossEntropyLoss()

print("Optimizer : Adam")
print("Learning rate : 0.0001 (10x smaller than Day 4)")
print("Scheduler : ReduceLROnPlateau")
print()
print("Scheduler means: if val accuracy stops improving")
print("for 2 epochs, cut learning rate in half automatically")

Optimizer : Adam
Learning rate : 0.0001 (10x smaller than Day 4)
Scheduler : ReduceLROnPlateau

Scheduler means: if val accuracy stops improving
for 2 epochs, cut learning rate in half automatically


In [4]:
num_epochs = 10
best_val_acc = 0.0
device = torch.device("cpu")
model = model.to(device)

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0
    train_correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()

    # Validation phase
    model.eval()
    val_loss = 0
    val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()

    train_acc = 100 * train_correct / len(train_dataset)
    val_acc   = 100 * val_correct / len(val_dataset)

    # Save best model automatically
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "clearbin_best.pth")
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Acc: {train_acc:.1f}% | "
              f"Val Acc: {val_acc:.1f}% ← best saved!")
    else:
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Acc: {train_acc:.1f}% | "
              f"Val Acc: {val_acc:.1f}%")

    scheduler.step(val_acc)

print(f"\nFine tuning complete!")
print(f"Best val accuracy: {best_val_acc:.1f}%")

Epoch 1/10 | Train Acc: 83.9% | Val Acc: 86.2% ← best saved!
Epoch 2/10 | Train Acc: 89.6% | Val Acc: 89.4% ← best saved!
Epoch 3/10 | Train Acc: 94.2% | Val Acc: 88.8%
Epoch 4/10 | Train Acc: 96.0% | Val Acc: 90.6% ← best saved!
Epoch 5/10 | Train Acc: 97.2% | Val Acc: 90.7% ← best saved!
Epoch 6/10 | Train Acc: 97.9% | Val Acc: 91.9% ← best saved!
Epoch 7/10 | Train Acc: 98.2% | Val Acc: 91.3%
Epoch 8/10 | Train Acc: 98.9% | Val Acc: 92.1% ← best saved!
Epoch 9/10 | Train Acc: 98.9% | Val Acc: 91.5%
Epoch 10/10 | Train Acc: 98.9% | Val Acc: 92.7% ← best saved!

Fine tuning complete!
Best val accuracy: 92.7%


In [5]:
# ============================================
# CLEARBIN - DAY 5 NOTES
# ============================================

# 1. Fine Tuning vs Transfer Learning
#    - Day 4: Only last layer trained (7,686 params)
#    - Day 5: All layers unfrozen (4,015,234 params)
#    - Must load Day 4 weights first before unfreezing
#    - Unfreezing randomly = destroys pretrained knowledge

# 2. Learning Rate for Fine Tuning
#    - Day 4 LR : 0.001  (learning from scratch)
#    - Day 5 LR : 0.0001 (careful fine tuning)
#    - 10x smaller = tiny careful steps

# 3. ReduceLROnPlateau Scheduler
#    - Monitors val accuracy every epoch
#    - If no improvement for 2 epochs → LR cut in half
#    - Helps squeeze out last few % of accuracy

# 4. Best Model Saving
#    - Save only when val accuracy improves
#    - Prevents saving an overfit model
#    - clearbin_best.pth = our final model

# 5. Final Results
#    - Best val accuracy : 92.7%
#    - Train accuracy    : 98.9%
#    - Small gap = slight overfitting but acceptable

print("Day 5 complete! 92.7% accuracy achieved!")

Day 5 complete! 92.7% accuracy achieved!
